In [ ]:
!pip install -q ipywidgets

In [ ]:
import os
import glob
import time
import math
import random

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import GradScaler, autocast
from tqdm import tqdm

torch.backends.cudnn.benchmark = True

## # CONFIG

In [ ]:
DATASET_DIR   = "./conversational_dataset"
SAMPLE_RATE   = 16000
CROP_SECONDS  = 4
CROP_SAMPLES  = SAMPLE_RATE * CROP_SECONDS

BATCH_SIZE    = 16
EPOCHS        = 40
LR            = 1e-3
WEIGHT_DECAY  = 1e-2
WARMUP_STEPS  = 400
CLIP_GRAD     = 5.0
STFT_LAMBDA_MAX    = 0.1
STFT_LAMBDA_FLOOR  = 0.01
STFT_WARMUP_START  = 8
STFT_WARMUP_END    = 15
UNROLL_PROB        = 0.5
UNROLL_START_EPOCH = 20
CONSISTENCY_WEIGHT = 0.1
NUM_WORKERS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_DIR    = "./checkpoints"
RESUME_CHECKPOINT = None

ENC_KERNEL    = 16
ENC_STRIDE    = 8
ENC_FILTERS   = 64
DPRNN_HIDDEN  = 128
DPRNN_LAYERS  = 6
CHUNK_SIZE    = 100

RMS_STOP_THRESHOLD  = 0.01
MAX_RECURSIVE_ITERS = 6

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## # DATASET

In [ ]:
class SeparationDataset(Dataset):

    def __init__(self, root, split):
        self.crop     = CROP_SAMPLES
        self.is_train = (split == "train")

        sample_dirs  = sorted(glob.glob(os.path.join(root, split, "sample_*")))
        
        self.index = []
        self.speaker_counts = []
        for d in sample_dirs:
            sp = glob.glob(os.path.join(d, "source_*.flac"))
            if len(sp) >= 2:
                self.index.append(d)
                self.speaker_counts.append(len(sp))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sample_dir   = self.index[idx]
        source_paths = sorted(glob.glob(os.path.join(sample_dir, "source_*.flac")))
        n            = len(source_paths)

        sources = []
        for sp in source_paths:
            audio, _ = sf.read(sp, dtype="float32")
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            sources.append(audio)

        length  = min(len(s) for s in sources)
        sources = [s[:length] for s in sources]

        if length < self.crop:
            repeats = math.ceil(self.crop / length)
            sources = [np.tile(s, repeats)[:self.crop] for s in sources]
        elif self.is_train:
            start   = random.randint(0, length - self.crop)
            sources = [s[start:start + self.crop] for s in sources]
        else:
            start   = (length - self.crop) // 2
            sources = [s[start:start + self.crop] for s in sources]

        sources    = np.stack(sources, axis=0)
        target_idx = random.randint(0, n - 1) if self.is_train else idx % n
        target     = sources[target_idx]
        residual   = sources[np.arange(n) != target_idx].sum(axis=0)
        mixture    = sources.sum(axis=0)

        if self.is_train:
            max_speakers = 4
            padded_sources = np.zeros((max_speakers, self.crop), dtype=np.float32)
            padded_sources[:n] = sources
            return (
                torch.from_numpy(mixture).float(),
                torch.from_numpy(target).float(),
                torch.from_numpy(residual).float(),
                torch.from_numpy(padded_sources).float(),
                target_idx,
                n
            )
        else:
            return (
                torch.from_numpy(mixture).float(),
                torch.from_numpy(target).float(),
                torch.from_numpy(residual).float(),
            )

class CurriculumSampler(torch.utils.data.Sampler):
    def __init__(self, dataset):
        self.dataset = dataset
        self.num_samples = len(dataset)
        self.weights = torch.ones(self.num_samples)
        
    def set_weights(self, weights):
        self.weights = weights
        
    def __iter__(self):
        return iter(torch.multinomial(self.weights, self.num_samples, replacement=True).tolist())
        
    def __len__(self):
        return self.num_samples

def get_curriculum_weights(epoch, total_epochs, speaker_counts):
    weights = []
    if epoch <= 10:
        w34 = 0.0
    elif epoch <= 20:
        w34 = (epoch - 10) / 10.0
    else:
        w34 = 1.0
        
    for n in speaker_counts:
        if n == 2:
            weights.append(1.0)
        else:
            weights.append(w34)
            
    return torch.tensor(weights, dtype=torch.float)

def get_loaders():
    train_ds = SeparationDataset(DATASET_DIR, "train")
    val_ds   = SeparationDataset(DATASET_DIR, "val")

    base_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        base_kwargs.update(persistent_workers=True, prefetch_factor=2)

    train_sampler = CurriculumSampler(train_ds)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, drop_last=True, **base_kwargs)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 4, shuffle=False,                 **base_kwargs)

    return train_dl, val_dl, train_ds

## # MODEL

In [ ]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(1, ENC_FILTERS, ENC_KERNEL, stride=ENC_STRIDE, bias=False)

    def forward(self, x):
        return F.relu(self.conv(x))


class Decoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(ENC_FILTERS, 1, ENC_KERNEL, stride=ENC_STRIDE, bias=False)

    def forward(self, x):
        return self.deconv(x)


class DPRNN_Block(nn.Module):

    def __init__(self, feat_dim, hidden):
        super().__init__()
        self.intra_rnn = nn.LSTM(feat_dim, hidden, batch_first=True, bidirectional=True)
        self.intra_fc  = nn.Linear(hidden * 2, feat_dim)
        self.intra_ln  = nn.LayerNorm(feat_dim)

        self.inter_rnn = nn.LSTM(feat_dim, hidden, batch_first=True, bidirectional=True)
        self.inter_fc  = nn.Linear(hidden * 2, feat_dim)
        self.inter_ln  = nn.LayerNorm(feat_dim)

    def forward(self, x):
        B, N, K, S = x.shape

        intra = x.permute(0, 3, 2, 1).contiguous().view(B * S, K, N)
        out, _ = self.intra_rnn(intra)
        out    = self.intra_fc(out).view(B, S, K, N).permute(0, 3, 2, 1)
        x      = self.intra_ln((x + out).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

        inter = x.permute(0, 2, 3, 1).contiguous().view(B * K, S, N)
        out, _ = self.inter_rnn(inter)
        out    = self.inter_fc(out).view(B, K, S, N).permute(0, 3, 1, 2)
        x      = self.inter_ln((x + out).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

        return x


class DPRNN_Separator(nn.Module):

    def __init__(self):
        super().__init__()
        self.norm       = nn.GroupNorm(1, ENC_FILTERS)
        self.input_proj = nn.Conv1d(ENC_FILTERS, ENC_FILTERS, 1)
        self.blocks     = nn.ModuleList([DPRNN_Block(ENC_FILTERS, DPRNN_HIDDEN) for _ in range(DPRNN_LAYERS)])
        self.mask_net   = nn.Sequential(nn.PReLU(), nn.Conv2d(ENC_FILTERS, 2 * ENC_FILTERS, 1))

    def forward(self, enc):
        B, N, L = enc.shape
        x       = self.input_proj(self.norm(enc))

        chunk   = CHUNK_SIZE
        hop     = chunk // 2
        pad_len = (hop - (L % hop)) % hop

        if pad_len:
            x = F.pad(x, (0, pad_len))
        L_pad = x.shape[2]

        # Vectorized chunk extraction: [B, N, L_pad] -> [B, N, num_chunks, chunk] -> [B, N, chunk, num_chunks]
        chunks = x.unfold(2, chunk, hop).permute(0, 1, 3, 2).contiguous()

        for block in self.blocks:
            chunks = block(chunks)

        # masks: [B, 2*N, chunk, num_chunks]
        masks = torch.sigmoid(self.mask_net(chunks))
        _, _, K, S = masks.shape
        masks = masks.view(B, 2, N, K, S)

        # Vectorized overlap-add via F.fold
        masks_r = masks.reshape(B * 2, N * K, S)
        out     = F.fold(masks_r, output_size=(1, L_pad), kernel_size=(1, K), stride=(1, hop))
        count   = F.fold(torch.ones_like(masks_r), output_size=(1, L_pad), kernel_size=(1, K), stride=(1, hop))
        out     = (out / count.clamp(min=1)).view(B, 2, N, L_pad)[:, :, :, :L]

        return out * enc.unsqueeze(1)


class ORPIT_Model(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder   = Encoder()
        self.separator = DPRNN_Separator()
        self.decoder   = Decoder()

    def forward(self, mixture):
        B, T    = mixture.shape
        mix_enc = self.encoder(mixture.unsqueeze(1))
        sep_enc = self.separator(mix_enc)

        out = self.decoder(sep_enc.view(B * 2, ENC_FILTERS, -1)).view(B, 2, -1)
        return out[:, :, :T]

## # LOSS

In [ ]:
_hann_windows = {}

def get_hann_window(size, device):
    key = (size, str(device))
    if key not in _hann_windows:
        _hann_windows[key] = torch.hann_window(size, device=device)
    return _hann_windows[key]

def si_sdr(est, ref, eps=1e-8):
    ref    = ref - ref.mean(dim=-1, keepdim=True)
    est    = est - est.mean(dim=-1, keepdim=True)
    alpha  = (ref * est).sum(-1, keepdim=True) / (ref.pow(2).sum(-1, keepdim=True) + eps)
    signal = alpha * ref
    noise  = est - signal
    return 10 * torch.log10(signal.pow(2).sum(-1) / (noise.pow(2).sum(-1) + eps) + eps)

def stft_loss_single(est, ref, fft_size):
    hop    = fft_size // 4
    win    = fft_size // 2
    window = get_hann_window(win, est.device)

    est_mag = torch.stft(est, fft_size, hop, win, window, return_complex=True).abs().clamp(min=1e-8)
    ref_mag = torch.stft(ref, fft_size, hop, win, window, return_complex=True).abs().clamp(min=1e-8)

    sc = torch.norm(ref_mag - est_mag, "fro") / torch.norm(ref_mag, "fro")
    lm = F.l1_loss(torch.log(est_mag), torch.log(ref_mag))
    return sc + lm

def multi_res_stft_loss(est, ref):
    est_f = est.float().reshape(-1, est.shape[-1])
    ref_f = ref.float().reshape(-1, ref.shape[-1])
    return sum(stft_loss_single(est_f, ref_f, fft) for fft in [512, 1024, 2048]) / 3

def pit_loss(out, target_ref, residual_ref, mixture, stft_weight):
    out1, out2 = out[:, 0], out[:, 1]

    sisdr_a = si_sdr(out1, target_ref)   + si_sdr(out2, residual_ref)
    sisdr_b = si_sdr(out1, residual_ref) + si_sdr(out2, target_ref)

    use_a        = (sisdr_a >= sisdr_b)
    target_est   = torch.where(use_a.unsqueeze(-1), out1, out2)
    residual_est = torch.where(use_a.unsqueeze(-1), out2, out1)
    best_sisdr   = torch.where(use_a, sisdr_a, sisdr_b) / 2

    if stft_weight > 0:
        stft = multi_res_stft_loss(target_est, target_ref) + multi_res_stft_loss(residual_est, residual_ref)
    else:
        stft = 0.0

    consistency = F.l1_loss(target_est + residual_est, mixture)

    loss = (-best_sisdr + stft_weight * stft).mean() + CONSISTENCY_WEIGHT * consistency
    
    stft_val = stft.item() if isinstance(stft, torch.Tensor) else stft
    metrics = {
        "sisdr": best_sisdr.mean().item(),
        "stft": stft_val,
        "consistency": consistency.item()
    }
    return loss, metrics, target_est, residual_est

## # TRAINING

In [ ]:
def run_epoch(model, loader, optimizer, scaler, is_train, stft_weight, epoch=0, global_step=0):
    model.train() if is_train else model.eval()

    n = len(loader)
    assert n > 0, "DataLoader has 0 batches"

    total_loss  = 0.0
    total_metrics = {"sisdr": 0.0, "stft": 0.0, "consistency": 0.0}
    ctx         = torch.enable_grad if is_train else torch.no_grad
    unroll_batches = 0

    with ctx():
        pbar = tqdm(loader, leave=True, desc="train" if is_train else "val  ")
        for batch in pbar:
            if is_train:
                mix, target_ref, residual_ref, sources, target_idx, num_speakers = batch
                sources = sources.to(DEVICE)
                target_idx = target_idx.to(DEVICE)
                num_speakers = num_speakers.to(DEVICE)
            else:
                mix, target_ref, residual_ref = batch

            mix          = mix.to(DEVICE)
            target_ref   = target_ref.to(DEVICE)
            residual_ref = residual_ref.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                out = model(mix)
                loss, metrics, target_est, residual_est = pit_loss(out, target_ref, residual_ref, mix, stft_weight)

                if is_train and epoch >= UNROLL_START_EPOCH and random.random() < UNROLL_PROB:
                    unroll_batches += 1
                    mix2 = residual_est.detach()
                    
                    B = mix.shape[0]
                    target_ref2 = torch.zeros_like(target_ref)
                    residual_ref2 = torch.zeros_like(residual_ref)
                    
                    for b in range(B):
                        n_spk = num_speakers[b].item()
                        avail_idx = [i for i in range(n_spk) if i != target_idx[b].item()]
                        if len(avail_idx) > 0:
                            new_target_idx = random.choice(avail_idx)
                            target_ref2[b] = sources[b, new_target_idx]
                            
                            rem_idx = [i for i in avail_idx if i != new_target_idx]
                            if len(rem_idx) > 0:
                                residual_ref2[b] = sources[b, rem_idx].sum(dim=0)
                            
                    out2 = model(mix2)
                    loss2, metrics2, _, _ = pit_loss(out2, target_ref2, residual_ref2, mix2, stft_weight)
                    loss = loss + 0.5 * loss2
                    for k in metrics:
                        metrics[k] = metrics[k] + 0.5 * metrics2[k]

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
                scaler.step(optimizer)
                scaler.update()

                global_step += 1
                if global_step <= WARMUP_STEPS:
                    current_lr = LR * (global_step / WARMUP_STEPS)
                    for pg in optimizer.param_groups:
                        pg["lr"] = current_lr

            total_loss  += loss.item()
            for k in metrics:
                total_metrics[k] += metrics[k]
                
            pbar.set_postfix(
                loss=f"{loss.item():.3f}", 
                sisdr=f"{metrics['sisdr']:.2f}",
                stft=f"{metrics['stft']:.3f}",
                cons=f"{metrics['consistency']:.3f}"
            )

    return total_loss / n, {k: v / n for k, v in total_metrics.items()}, global_step, unroll_batches


def train():
    train_dl, val_dl, train_ds = get_loaders()

    model = ORPIT_Model().to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {total_params / 1e6:.2f}M")
    print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

    from torch.optim import AdamW
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler    = GradScaler("cuda") if DEVICE == "cuda" else GradScaler("cpu")

    start_epoch   = 1
    best_val_loss = float("inf")
    global_step   = 0

    if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
        ckpt  = torch.load(RESUME_CHECKPOINT, map_location=DEVICE)
        state = {k.replace("module.", ""): v for k, v in ckpt["model"].items()}

        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(state)
        else:
            model.load_state_dict(state)

        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch   = ckpt["epoch"] + 1
        best_val_loss = ckpt["best_val_loss"]
        global_step   = ckpt.get("global_step", 0)
        print(f"Resumed from epoch {ckpt['epoch']}")

    if start_epoch > EPOCHS:
        print(f"Already at {EPOCHS} epochs — nothing to train.")
        return model

    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()

        weights = get_curriculum_weights(epoch, EPOCHS, train_ds.speaker_counts)
        train_dl.sampler.set_weights(weights)

        if epoch <= STFT_WARMUP_START:
            stft_weight = STFT_LAMBDA_FLOOR
        elif epoch <= STFT_WARMUP_END:
            stft_weight = STFT_LAMBDA_FLOOR + (STFT_LAMBDA_MAX - STFT_LAMBDA_FLOOR) * (epoch - STFT_WARMUP_START) / (STFT_WARMUP_END - STFT_WARMUP_START)
        else:
            stft_weight = STFT_LAMBDA_MAX

        train_loss, train_metrics, global_step, unroll_batches = run_epoch(
            model, train_dl, optimizer, scaler, is_train=True, stft_weight=stft_weight, epoch=epoch, global_step=global_step
        )
        
        run_val = (epoch % 5 == 0) or (epoch == EPOCHS) or (epoch == 1)
        
        if run_val:
            val_loss, val_metrics, _, _ = run_epoch(
                model, val_dl, optimizer, scaler, is_train=False, stft_weight=stft_weight, epoch=epoch, global_step=global_step
            )
            val_sisdr = val_metrics["sisdr"]
            if global_step > WARMUP_STEPS:
                scheduler.step(-val_sisdr)

        elapsed = time.time() - t0
        lr      = optimizer.param_groups[0]["lr"]

        unroll_frac = unroll_batches / len(train_dl) if len(train_dl) > 0 else 0
        unroll_str = f" | Unroll {unroll_frac:.0%}" if unroll_batches > 0 else ""

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | {elapsed:.0f}s | stft_w={stft_weight:.3f}{unroll_str} | LR {lr:.2e}",
            f"  Train: Loss {train_loss:.4f} | SI-SDR {train_metrics['sisdr']:.2f} | STFT {train_metrics['stft']:.3f} | Cons {train_metrics['consistency']:.3f}"
        )
        if run_val:
            print(f"  Val  : Loss {val_loss:.4f} | SI-SDR {val_metrics['sisdr']:.2f} | STFT {val_metrics['stft']:.3f} | Cons {val_metrics['consistency']:.3f}")
        else:
            print(f"  Val  : Skipped (Runs every 5 epochs)")

        last_path = os.path.join(CHECKPOINT_DIR, "last.pt")
        if os.path.exists(last_path):
            os.remove(last_path)

        ckpt = {
            "epoch": epoch, "model": model.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "best_val_loss": best_val_loss,
            "global_step": global_step
        }
        torch.save(ckpt, last_path)

        if run_val and -val_sisdr < best_val_loss:
            best_val_loss = -val_sisdr
            ckpt = {
                "epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "best_val_loss": best_val_loss,
                "global_step": global_step
            }
            torch.save(ckpt, os.path.join(CHECKPOINT_DIR, "best.pt"))
            print(f"  --> New best saved ({best_val_loss:.4f})")

    return model

## # INFERENCE

In [ ]:
def load_model_for_inference(checkpoint_path):
    model = ORPIT_Model().to(DEVICE)
    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    state = ckpt["model"] if "model" in ckpt else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval()
    return model


def rms(signal):
    return signal.pow(2).mean().sqrt().item()


def recursive_separate(model, mixture_np, rms_threshold=RMS_STOP_THRESHOLD, max_iters=MAX_RECURSIVE_ITERS):
    mix      = torch.from_numpy(mixture_np).float().unsqueeze(0).to(DEVICE)
    speakers = []

    with torch.no_grad():
        for _ in range(max_iters):
            out      = model(mix)
            target   = out[0, 0]
            residual = out[0, 1]

            speakers.append(target.cpu().numpy())

            if rms(residual) < rms_threshold:
                break

            mix = residual.unsqueeze(0)

    return speakers



model = train()